# HelpDesk-AI : carnet de prise en main

Mission : audit de sécurité (boîte grise) de l'agent d'assistance informatique de **Helvetia Systems SA**.

Ce carnet montre **uniquement l'API** de la cible. Il **ne contient aucune piste d'exploitation** : c'est à vous de concevoir vos tests, dans le respect des **Règles d'engagement** (`regles_engagement_FR.md`).

**Prérequis** (voir `INSTALL_FR.md`) : `uv sync`, puis Ollama démarré avec les modèles `qwen2.5:3b` et `nomic-embed-text`.

Documents à lire avant de commencer : `lettre_de_mission_FR.md`, `regles_engagement_FR.md`, `doc_client_FR.md`, `modele_rapport_FR.md`.

## 1. Instancier l'agent

`HelpdeskAgent.for_team()` charge la configuration **de votre équipe** (jetons de preuve et vulnérabilités actives compris, sans que vous en connaissiez la liste).

In [ ]:
from helpdesk_agent import HelpdeskAgent

agent = HelpdeskAgent.for_team()
print(agent.chat("Bonjour, comment réinitialiser mon mot de passe ?"))

## 2. Dialoguer avec l'agent : `chat`

`agent.chat(message) -> str` envoie une demande et renvoie la réponse. L'agent peut appeler ses outils internes (voir `doc_client_FR.md`, §4) pour répondre.

In [ ]:
print(agent.chat("Je n'arrive pas à me connecter au VPN, que faire ?"))

## 3. Rendre une réponse dans le portail : `render`

`agent.render(text) -> str` produit le HTML tel que le **portail** support l'afficherait. Le portail se comporte comme un navigateur.

In [ ]:
html = agent.render("**Bonjour**, voici la procédure demandée.")
print(html)

## 4. Déposer un ticket et traiter la file : `submit_ticket`, `process_ticket_queue`

`submit_ticket(body, user=..., subject=...) -> ticket_id` dépose un ticket.
`process_ticket_queue() -> list[dict]` fait traiter **de façon autonome** tous les tickets ouverts par l'agent.

In [ ]:
tid = agent.submit_ticket(
    body="Mon imprimante est bloquée, un bourrage papier.",
    user="a.employe", subject="Imprimante en panne")
print("ticket déposé :", tid)

resultats = agent.process_ticket_queue()
for r in resultats:
    print(r["ticket_id"], "->", r["reply"][:200])

## 5. Proposer un article de base de connaissances : `suggest_kb`

`agent.suggest_kb(title, text) -> str` soumet une proposition d'article. La réponse indique le statut du traitement.

In [ ]:
print(agent.suggest_kb("Titre proposé", "Contenu proposé pour la base de connaissances."))

## 6. Points d'observation (instrumentation)

Le kit expose des observateurs qui **simulent** les effets de bord de la production. Ils servent à la preuve et à la reproduction (voir `doc_client_FR.md`).

- `agent.outbox` : courriels « envoyés » (boîte d'envoi simulée)
- `agent.usage_alerts` : alertes du compteur d'usage
- `agent.attacker.log` : journal du serveur externe contrôlé par l'attaquant (simulé, en mémoire)
- `agent.renderer.findings` : constats du moteur de rendu du portail

In [ ]:
print("outbox        :", agent.outbox)
print("usage_alerts  :", agent.usage_alerts)
print("attacker.log  :", agent.attacker.log)
print("render finds  :", agent.renderer.findings)

## 7. À vous de jouer

Vous connaissez maintenant toute la surface d'API. Concevez vos tests d'attaque, produisez les jetons `CPE{…}` par exploitation applicative, et documentez chaque finding dans `modele_rapport_FR.md`.

**Rappels :** restez dans le périmètre des Règles d'engagement (pas de rétro-ingénierie du paquet, pas de lecture de `team.cfg`, pas d'attaque d'Ollama). Un finding n'est confirmé que si son jeton est reproductible. Les résultats négatifs sont attendus et notés.